# B1.7 Beyond Euler Angles: Quaternions, Axis-Angle, and Rotation Vectors for 3D Orientation

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.1 Learning Objectives

By the end of this lesson, you will be able to:

- Understand why Euler angles are limited for robust 3D orientation
- Define and compare quaternions, axis-angle, and rotation vectors
- Rotate vectors using quaternions
- Compose multiple 3D rotations with quaternion multiplication
- Convert between all major rotation representations
- Choose the right representation for use in physics, simulation, robotics, or graphics

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.2 Why Look Beyond Euler Angles?

Euler angles — typically written as a sequence like $(\phi, \theta, \psi)$ — are widely used because they align well with our human intuition: we think in terms of pitch, yaw, and roll.

However, Euler angles suffer from **three critical limitations**:

1. **Gimbal Lock**: When two rotation axes align, one degree of freedom is lost. This makes smooth or arbitrary rotation problematic.
2. **Order Sensitivity**: Changing the order of applying Euler rotations changes the result — it's not commutative.
3. **Discontinuities**: Interpolating between two orientations represented as Euler angles can lead to awkward jumps or flips.

To overcome these issues, we use **rotation representations** that are:

- Geometrically meaningful
- Free from singularities
- Smooth and continuous
- Efficient for computation and storage

These include:

- **Quaternions**: the go-to solution in aerospace, robotics, and game engines
- **Axis-angle**: geometrically intuitive and compact
- **Rotation vectors**: minimal and common in optimization

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.3 Rotation Representations Overview

| Representation     | Parameters           | Key Advantages                          | Main Limitations                |
|--------------------|----------------------|------------------------------------------|----------------------------------|
| Euler Angles       | 3 angles              | Easy to understand; minimal              | Gimbal lock; not robust         |
| Rotation Matrix    | 3×3 = 9 values        | Linear algebra compatible                | Redundant; non-minimal          |
| Quaternion         | 4 values              | Compact, no gimbal lock, smooth SLERP    | Less intuitive                  |
| Axis-Angle         | 1 unit vector + 1 angle | Geometrically meaningful                | Requires conversion to apply    |
| Rotation Vector    | 3 values              | Minimal version of axis-angle            | Slightly less intuitive         |

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.4 Axis-Angle Representation

### What is it?

Every 3D rotation can be described by **rotating around a single axis** by a given angle. This is the basis of **axis-angle** representation.

It uses:
- A **unit vector** $\hat{n} = (n_x, n_y, n_z)$: the **axis** of rotation
- An **angle** $\theta$: the **amount** of rotation around that axis

Together, this describes any 3D rotation unambiguously and intuitively.

**Example**: “Rotate 90° around the X-axis” →  
$\hat{n} = (1, 0, 0)$, $\theta = \frac{\pi}{2}$

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.5 Rotation Vector

The **rotation vector** is a compressed version of axis-angle:

$$
\vec{\theta} = \theta \, \hat{n}
$$

- It encodes **direction** (axis) and **magnitude** (angle) in a single 3D vector.
- It's used in applications requiring compact or differentiable representations (robotics, computer vision, optimization).

✅ Efficient to store  
✅ Easy to linearize for small-angle approximations  
✅ Used in `cv2.Rodrigues()` in OpenCV

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.6 Quaternions for 3D Rotation 


### What Is a Quaternion?

A **quaternion** is an extension of complex numbers to **four dimensions**, developed by William Rowan Hamilton in 1843.

A quaternion is typically written as:

$$
q = w + xi + yj + zk
$$

Where:

- $w$: the **scalar part** (real number)
- $(x, y, z)$: the **vector part** (imaginary components)
- $i, j, k$: the **imaginary units** of quaternion algebra


### The Imaginary Units: $i$, $j$, and $k$

These units behave differently from real numbers:

- $i^2 = j^2 = k^2 = -1$
- They obey **non-commutative multiplication rules**:

  $$
  ij = k, \quad jk = i, \quad ki = j \\
  ji = -k, \quad kj = -i, \quad ik = -j
  $$

This structure allows quaternions to **encode rotations**, where the vector part behaves like an axis of rotation and the scalar part encodes the angle.


### Component View

We usually represent quaternions as a 4D vector:

$$
q = [w, x, y, z]
$$

Or:

- $w$: **scalar part**, related to the angle of rotation
- $(x, y, z)$: **vector part**, points along the **axis of rotation**

Together, they form a **rotation** when the quaternion has **unit length**:


### What Is a Unit Quaternion?

A **unit quaternion** has a norm (magnitude) equal to 1:

$$
|q| = \sqrt{w^2 + x^2 + y^2 + z^2} = 1
$$

Only **unit quaternions** represent **pure 3D rotations**.

Non-unit quaternions can represent scaling or distortion — but in rotational dynamics, we use **unit quaternions exclusively** to describe orientation.

Just like unit vectors describe direction only (not magnitude), unit quaternions describe **orientation only**, not how far something rotates.



### Using a Quaternion to Represent a Rotation

To represent a rotation of angle $\theta$ about a **unit axis** $\hat{n} = (n_x, n_y, n_z)$:

$$
q = \left[\cos \frac{\theta}{2},\,
n_x \sin \frac{\theta}{2},\,
n_y \sin \frac{\theta}{2},\,
n_z \sin \frac{\theta}{2} \right]
$$

Interpretation:

- $\cos(\theta/2)$: scalar part → encodes the **rotation amount**
- $\hat{n} \sin(\theta/2)$: vector part → encodes the **axis and direction**

So a quaternion compactly encodes both the **rotation axis and rotation angle**.


### Rotating a Vector Using a Quaternion

To rotate a 3D vector $\vec{v}$ using a quaternion $q$:

1. Convert $\vec{v}$ into a **pure quaternion**:

   $$
   v_q = [0,\, \vec{v}] = [0,\, v_x, v_y, v_z]
   $$

2. Compute the rotated vector using:

   $$
   v_{\text{rotated}} = q \cdot v_q \cdot q^{-1}
   $$

This uses **quaternion multiplication** (not matrix multiplication). The result is another quaternion — its **vector part** is the rotated vector.


### Visual Interpretation

- The **axis** of rotation is the direction of the vector part $(x, y, z)$.
- The **amount** of rotation is encoded in the scalar $w$:
  - $w = \cos(\theta/2)$, so $\theta = 2 \cos^{-1}(w)$
- The quaternion essentially **“twists” space** around its vector part


### Summary: Quaternion Anatomy

| Component     | Symbol     | Meaning                             |
|---------------|------------|--------------------------------------|
| Scalar part   | $w$        | Related to rotation angle $\theta$ via $w = \cos(\theta/2)$ |
| Vector part   | $(x, y, z)$| Direction of axis × $\sin(\theta/2)$ |
| Full quaternion | $[w, x, y, z]$ | Encodes axis-angle rotation |
| Norm          | $|q|$      | Must be 1 for valid rotation        |
| Conjugate     | $q^* = [w, -x, -y, -z]$ | Used for inverse rotation |
| Inverse       | $q^{-1} = q^*$ (if unit) | Rotation in opposite direction |



This makes quaternions ideal for:

- 3D simulations
- Orientation tracking
- Smooth interpolation (SLERP)
- Spacecraft and drone attitude control
- Anywhere rotation must be robust, continuous, and gimbal-lock-free


### From Axis-Angle to Quaternion

Given axis $\hat{n} = (n_x, n_y, n_z)$ and angle $\theta$:

$$
q = \left[\cos \frac{\theta}{2},\, n_x \sin \frac{\theta}{2},\, n_y \sin \frac{\theta}{2},\, n_z \sin \frac{\theta}{2} \right]
$$

This quaternion represents rotation around $\hat{n}$ by angle $\theta$.



### From Quaternion to Axis-Angle / Rotation Vector

Given a unit quaternion $q = [w, x, y, z]$:

- Compute angle:
  $$
  \theta = 2 \cos^{-1}(w)
  $$

- Compute axis:
  $$
  \hat{n} = \frac{(x, y, z)}{\sin(\theta/2)} \quad \text{(if } \sin(\theta/2) \ne 0 \text{)}
  $$

- Then rotation vector:
  $$
  \vec{\theta} = \theta \hat{n}
  $$

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.7 Quaternion Operations

### Rotating a Vector

To rotate a vector $\vec{v}$ using a unit quaternion $q$:

1. Convert to a pure quaternion:
   $$
   v_q = [0, \vec{v}]
   $$

2. Apply rotation:
   $$
   v_{\text{rotated}} = q \cdot v_q \cdot q^{-1}
   $$

The result’s **vector part** is the rotated vector.


### Composing Rotations

If $q_1$ and $q_2$ are unit quaternions representing two rotations:

$$
q_{\text{combined}} = q_2 \cdot q_1
$$

- First apply $q_1$, then $q_2$
- Quaternion multiplication is **non-commutative**

This is **cleaner and faster** than composing rotation matrices or Euler angles.

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.8 Euler Angles to Quaternion (ZXZ)

In the **ZXZ convention**, to convert Euler angles $(\phi, \theta, \psi)$ to quaternion:

1. Construct elementary quaternions:

- $q_z(\phi) = \left[\cos \frac{\phi}{2},\, 0, 0, \sin \frac{\phi}{2} \right]$
- $q_x(\theta) = \left[\cos \frac{\theta}{2},\, \sin \frac{\theta}{2}, 0, 0 \right]$
- $q_z(\psi) = \left[\cos \frac{\psi}{2},\, 0, 0, \sin \frac{\psi}{2} \right]$

2. Combine:

$$
q = q_z(\phi) \cdot q_x(\theta) \cdot q_z(\psi)
$$

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.9 Quaternion to Rotation Matrix

Given unit quaternion $q = [w, x, y, z]$, the corresponding rotation matrix is:

$$
R =
\begin{bmatrix}
1 - 2y^2 - 2z^2 & 2xy - 2zw & 2xz + 2yw \\
2xy + 2zw & 1 - 2x^2 - 2z^2 & 2yz - 2xw \\
2xz - 2yw & 2yz + 2xw & 1 - 2x^2 - 2y^2
\end{bmatrix}
$$

Use this if you need to apply the quaternion rotation to many vectors using matrix multiplication.

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.10 Comparison Summary

| Representation     | Components   | Intuitive? | Avoids Gimbal Lock | Composable | Interpolatable | Best For                         |
|--------------------|--------------|------------|---------------------|------------|----------------|----------------------------------|
| Euler Angles       | 3 angles     | ✅ Yes     | ❌ No               | ❌ No       | ❌ No          | Human input, basic UI            |
| Rotation Matrix    | 9 values     | ❌ Low     | ✅ Yes              | ✅ Yes      | ❌ Linear only | Physics, transforms, rendering   |
| Quaternion         | 4 values     | ⚠ Medium  | ✅ Yes              | ✅ Fast     | ✅ SLERP       | Robotics, animation, simulation  |
| Axis-Angle         | 1 axis + θ   | ✅ High    | ✅ Yes              | ✅ (via q)  | ✅ (via q)     | Geometry, visualization          |
| Rotation Vector    | 3 values     | ✅ Medium | ✅ Yes              | ✅ Yes      | ✅ Yes         | Optimization, CV, controls       |

<hr style="height:2px;border-width:0;color:gray;background-color:gray">

## B1.7.11 Final Insight

Euler angles are a great starting point — but:

- **Quaternions** are the most robust, smooth, and efficient rotation representation in 3D.
- **Axis-angle** is intuitive and connects to real-world physical rotation.
- **Rotation vectors** are compact and ideal for control, estimation, and optimization.

They all connect through the same geometry:

$$
\text{Rotation Vector} \leftrightarrow \text{Axis-Angle} \leftrightarrow \text{Quaternion} \leftrightarrow \text{Rotation Matrix}
$$

The right tool depends on the task: interpretability, interpolation, performance, or compactness.

<hr style="height:2px;border-width:0;color:gray;background-color:gray">